In [1]:
import time
import requests
import pandas as pd
from pathlib import Path
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

DATA_DIR = Path.cwd() / 'data'
if not DATA_DIR.exists():
    DATA_DIR = Path.cwd().parent / 'data'

cod = pd.read_csv(DATA_DIR / 'college_out_degree.csv')
print('college_out_degree shape:', cod.shape)
print('endow_total missing by year:')
print(cod.groupby('year')['col_endow_total'].apply(lambda x: x.isna().sum()))

college_out_degree shape: (5168, 21)
endow_total missing by year:
year
2019    2006
2023    2536
Name: col_endow_total, dtype: int64


In [2]:
session = requests.Session()
retries = Retry(
    total=8, connect=8, read=8, status=8,
    backoff_factor=0.75,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=frozenset(['GET']),
    respect_retry_after_header=True,
)
session.mount('https://', HTTPAdapter(max_retries=retries, pool_connections=10, pool_maxsize=10))
session.headers.update({'User-Agent': 'c2i-nacubo-endowments/1.0'})

def fetch_all(url):
    rows = []
    while url:
        for attempt in range(6):
            try:
                resp = session.get(url, timeout=(15, 120))
                resp.raise_for_status()
                data = resp.json()
                break
            except (requests.exceptions.ChunkedEncodingError, requests.exceptions.ConnectionError):
                if attempt == 5:
                    raise
                time.sleep(1.5 * (attempt + 1))
        rows.extend(data.get('results', []))
        url = data.get('next')
        time.sleep(0.05)
    return rows

In [3]:
# Pull 2022 endowment data (last available year from NACUBO endpoint)
# Note: 2022 was a down year for endowments (~18% market decline), so these values
# will slightly understate 2023 actuals — fine for relative comparisons, less so
# for absolute endowment levels in year-over-year analysis.
print('Pulling NACUBO endowments for 2022...')
rows = fetch_all('https://educationdata.urban.org/api/v1/college-university/nacubo/endowments/2022/')
print(f'  2022: {len(rows):,} rows')

endow_df = pd.DataFrame(rows)
endow_df = endow_df[['unitid', 'endow_total', 'endow_per_fte']].copy()
endow_df.rename(columns={'unitid': 'college_id'}, inplace=True)
endow_df['college_id'] = pd.to_numeric(endow_df['college_id'], errors='coerce')
endow_df['endow_total'] = pd.to_numeric(endow_df['endow_total'], errors='coerce')
endow_df['endow_per_fte'] = pd.to_numeric(endow_df['endow_per_fte'], errors='coerce')
endow_df = endow_df.dropna(subset=['college_id'])
endow_df['college_id'] = endow_df['college_id'].astype(int)
endow_df = endow_df.drop_duplicates(subset=['college_id'])

print(f'Unique colleges with 2022 endowment data: {len(endow_df):,}')
endow_df.head()

Pulling NACUBO endowments for 2022...
  2022: 664 rows
Unique colleges with 2022 endowment data: 664


,college_id,endow_total,endow_per_fte
0,100733,2.088711e+09,35194.885959
1,100858,1.079218e+09,38047.536436
2,101189,2.434041e+07,8259.385477
3,102094,2.373424e+08,20091.631338
4,102368,1.810742e+08,17266.543244


In [4]:
# Forward-fill: apply 2022 endowment values to 2023 rows that are missing
mask_2023 = cod['year'] == 2023

cod_2023 = cod[mask_2023].copy()
cod_2023 = cod_2023.merge(endow_df, on='college_id', how='left')

# Fill only where currently null
cod_2023['col_endow_total'] = cod_2023['col_endow_total'].fillna(cod_2023['endow_total'])
cod_2023['col_endow_per_fte'] = cod_2023['col_endow_per_fte'].fillna(cod_2023['endow_per_fte'])
cod_2023 = cod_2023.drop(columns=['endow_total', 'endow_per_fte'])

filled = cod_2023['col_endow_total'].notna().sum()
total = len(cod_2023)
print(f'2023 endow_total filled: {filled}/{total} ({filled/total*100:.1f}%)')

# Recombine with 2019 rows (unchanged)
cod_out = pd.concat([cod[cod['year'] == 2019], cod_2023], ignore_index=True)
cod_out = cod_out.sort_values(['year', 'college_id']).reset_index(drop=True)

print('\nendow_total missing by year (after fill):')
print(cod_out.groupby('year')['col_endow_total'].apply(lambda x: x.isna().sum()))

2023 endow_total filled: 609/2536 (24.0%)

endow_total missing by year (after fill):
year
2019    2006
2023    1927
Name: col_endow_total, dtype: int64


In [5]:
out_path = DATA_DIR / 'college_out_degree.csv'
cod_out.to_csv(out_path, index=False)
print(f'Saved: {out_path}  ({len(cod_out):,} rows)')

Saved: /Users/bryceclement/Desktop/c2i/data/college_out_degree.csv  (5,168 rows)
